# Task 4 - Comparison on Synthetic Non-Linear Model

**What it does, run top to bottom:**
sweep a set of measurement-noise levels `1/R²` (in dB); for each level generate one dataset and run
**two scenarios** — *Full Information* (the filter knows the true dynamics) and *Partial Information*
(the filter is given a mismatched evolution). For every (noise, scenario) it **trains every learned
model from scratch** and evaluates all five estimators on the *same* data:

| Method | Role | Trained how |
|---|---|---|
| EKF | classical baseline | — (analytic) |
| Robust EKF | classical robust baseline | — (tolerance `c` picked on training) |
| KalmanNet | learned KF gain | `Pipeline_EKF` |
| Original RT-KalmanNet | professor's MLP | `Pipeline_REKF` (our modular pipeline) |
| Proposed RT-KalmanNet | our GRU | `Pipeline_REKF` (our modular pipeline) |

## 1. Imports and project bootstrap

Locates the project root by walking up from the current directory until a `Simulations/` folder is
found, adds it to `sys.path`, and `chdir`s into it — so every relative import and file path below
resolves correctly regardless of where the notebook is launched from.

In [2]:
import os, sys, math, time, random, contextlib, io, json, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from tqdm.auto import tqdm

# find the project root (wherever Simulations/ lives) so paths work no matter
# where this notebook is opened from
_here = Path.cwd()
root = next((q for q in [_here, *_here.parents] if (q / "Simulations").is_dir()), _here)
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
os.chdir(root)

from Simulations import config
from Simulations.Synthetic_NL_model.parameters import Q_structure, R_structure, m, n, m1x_0, m2x_0, f, h
from Simulations.Extended_sysmdl import SystemModel
from Simulations.utils import DataGen
from Filters.EKF_test import EKFTest                                    # analytic EKF baseline
from RobustKalmanPY.robust_kalman_original import RobustKalman as Orig_RTKnet   # MLP, professor's
from RobustKalmanPY.robust_kalman_proposed import RobustKalman as Prop_RTKnet   # GRU, ours
from Pipelines.Pipeline_EKF import Pipeline_EKF                         # KalmanNet pipeline
from Pipelines.Pipeline_REKF import Pipeline_REKF                       # RT-KalmanNet pipeline
from KNet.KalmanNet_nn import KalmanNetNN

crit = nn.MSELoss(reduction="mean")

def to_dB(x):
    """Linear MSE -> dB."""
    return 10.0 * math.log10(float(x))

print("project root :", root)
print("torch        :", torch.__version__)
print(f"model        : synthetic non-linear, state dim m = {m}, obs dim n = {n}")

project root : c:\Users\Andrea\Documents\Università\Magistrale\Learning dynamical systems\RT-KalmanNet-project2026\CODE\RT_KFNET
torch        : 2.13.0+cpu
model        : synthetic non-linear, state dim m = 2, obs dim n = 2


## 2. Configuration (Edit only this cell)

Every learned model is trained from scratch for each (noise, scenario) pair, so the
total number of trainings is
`len(INV_R2_DB) × len(SCENARIOS) × (number of learned estimators enabled)`.

**The noise parameterisation** follows the KalmanNet paper's axis: $1/R^2$ in dB with $R = r^2 R_{\text{struct}}$ and $Q = q^2 Q_{\text{struct}}$.
The ratio $\nu = q^2/r^2$ is held **fixed** at `NU_DB`, so a single scalar moves both noises together
and the x-axis of every figure has one unambiguous meaning.

**Model selection: `RETRAIN`.** This single flag decides whether the learned estimators are trained
or reloaded, and it covers every (method, scenario, noise level) combination:

* `RETRAIN = True`: train every learned estimator, then save the checkpoints **and** the
  training/validation history to `training_history.json`.
* `RETRAIN = False`: load the saved checkpoints **and** the saved history; nothing is trained.
  Section 8 still draws the loss curves, and Sections 9–13 run normally.

Loading a model therefore does not lose the information about how it was trained.

> *Recommendation:* set `QUICK_RUN = True` for a preliminary end-to-end validation run — it shrinks
> the sweep to a single point, the dataset to a handful of short trajectories and the training to two
> epochs. This verifies that the whole notebook is wired correctly without incurring the full cost.

In [3]:
# User Configuration
SEED        = 0
USE_CUDA    = False          # in practice the linear-algebra ops here are CPU-only anyway
CKPT_DIR    = "Results_task4"   # kept separate so older runs don't get overwritten
RETRAIN     = False          # True: train from scratch, save checkpoints + history; False: load both
REGEN_DATA  = False           # True: (re)generate the datasets; False: reuse the cached ones
QUICK_RUN   = False           # True: fast run to check the code works

# Estimators to run
INCLUDE_EKF      = True      # classical baseline
INCLUDE_REKF     = True      # classical robust baseline (constant tolerance, grid search)
INCLUDE_KNET     = True      # KalmanNet (learned Kalman gain)
INCLUDE_ORIGINAL = True      # professor's MLP RT-KalmanNet
INCLUDE_PROPOSED = True      # our GRU RT-KalmanNet

# Experiment grid
INV_R2_DB   = [-12.04, -6.02, 0.0, 10.0]   # measurement noise sweep, 1/R^2 [dB]
NU_DB       = -20.0          # nu = q2/r2 [dB], held fixed across the sweep
SCENARIOS   = ["full", "partial"]          # full = true dynamics, partial = mismatched evolution

# Dataset parameters (one dataset of this size is generated per noise level)
N_E, N_CV, N_T = 120, 30, 30   # training / validation / test trajectories
T = 100                          # trajectory length, shared by all splits

# Filter initialization, shared by every estimator and both scenarios.
FILTER_P0   = 1e-3

# REKF baseline: constant-tolerance grid search
N_C_SEQ     = 6              # training sequences used to score each candidate tolerance
C_GRID      = np.geomspace(1e-4, 1.0, 20)   # candidate constant tolerances

N_TIME_SEQ  = 10              # test sequences timed one-by-one in the inference-time benchmark

# Model hyperparameters
KNET_MODEL = {"in_mult_KNet": 5, "out_mult_KNet": 40}
ORIG_MODEL = {"input_feat_mode": 3, "hidden_layers": [20, 20, 20, 20, 20]}
PROP_MODEL = {"input_feat_mode": 3, "gru_hidden_size": 64}

# Training hyperparameters
KNET_TRAIN = {"n_steps": 100, "n_batch": 10, "lr": 1e-3, "wd": 1e-4}
ORIG_TRAIN = {
    "n_steps": 60,           # total training epochs
    "n_batch": 8,
    "lr": 1e-4,
    "wd": 1e-4,
    "grad_clip": 1.0,        # max grad norm (None disables)
    "bptt_truncation": None, # window size for truncated BPTT (Proposed model only)
    "bptt_warmup_frac": None,# epoch fraction using truncated BPTT before full BPTT
}
PROP_TRAIN = {**ORIG_TRAIN, "bptt_truncation": 20, "bptt_warmup_frac": 0.4}

# Quick run overrides
if QUICK_RUN:
    INV_R2_DB, SCENARIOS = [0.0, 10.0], ["full", "partial"]
    N_E, N_CV, N_T, T = 20, 4, 4, 20
    N_C_SEQ = 2
    C_GRID = np.geomspace(1e-3, 1.0, 3)
    KNET_TRAIN = {**KNET_TRAIN, "n_steps": 20, "n_batch": 2}
    ORIG_TRAIN = {**ORIG_TRAIN, "n_steps": 2, "n_batch": 2}
    PROP_TRAIN = {**PROP_TRAIN, "n_steps": 2, "n_batch": 2, "bptt_truncation": T // 2}
    N_TIME_SEQ = 2

USE_CUDA = USE_CUDA and torch.cuda.is_available()
DEVICE = torch.device("cuda" if USE_CUDA else "cpu")
PATH_RESULTS = CKPT_DIR + "/"
DATA_DIR = os.path.join(CKPT_DIR, "datasets")
HIST_PATH = os.path.join(CKPT_DIR, "training_history.json")   # loss curves + training times
os.makedirs(DATA_DIR, exist_ok=True)

torch.manual_seed(SEED); np.random.seed(SEED); random.seed(SEED)
print(f"device = {DEVICE} | results -> {CKPT_DIR}/ | QUICK_RUN = {QUICK_RUN}")
print(f"sweep: 1/R^2 = {INV_R2_DB} dB  x  scenarios {SCENARIOS}   (nu = {NU_DB} dB)")
print(f"sizes: N_E = {N_E}, N_CV = {N_CV}, N_T = {N_T}, T = {T}")
print(f"RETRAIN = {RETRAIN} -> " + ("training from scratch; checkpoints + history will be saved"
                                    if RETRAIN else
                                    f"loading checkpoints + history ({HIST_PATH})"))

device = cpu | results -> Results_task4/ | QUICK_RUN = False
sweep: 1/R^2 = [-12.04, -6.02, 0.0, 10.0] dB  x  scenarios ['full', 'partial']   (nu = -20.0 dB)
sizes: N_E = 120, N_CV = 30, N_T = 30, T = 100
RETRAIN = False -> loading checkpoints + history (Results_task4\training_history.json)


## 3. Dataset (Generate, Split, Filter models)

One dataset per noise level, produced by `Simulations.utils.DataGen`, which draws the three splits
(train / validation / test) from the **true** synthetic non-linear model of
`Simulations/Synthetic_NL_model/parameters.py`:

$$x_{t+1} = \alpha\sin(\beta x_t + \varphi) + \delta + q_t,\qquad y_t = (x_t)^2 + r_t$$

* `train_y`, `cv_y`, `test_y` : observation sequences, shape `[N, n, T]`
* `train_x`, `cv_x`, `test_x` : the corresponding latent state sequences, shape `[N, m, T]`

### The two scenarios
**The data never changes between scenarios** — only the evolution function handed to the *filters*:

| Scenario | Filter's $f$ | Meaning |
|---|---|---|
| `full` | the true $\alpha\sin(\beta x + \varphi) + \delta$ | the filter knows the dynamics exactly |
| `partial` | $\sin(x)$ | the filter is given a mismatched evolution |

In [4]:
def f_partial(x):
    """Mismatched (partial-information) evolution given to the filter; data is still the true model's."""
    return torch.sin(x.clone())

# `args` carries the fields DataGen / EKFTest / Pipeline_EKF read out of config.py. Only Q and R
# change across the sweep, so the sizes below are set once and reused by every noise level.
args = config.general_settings()
args.use_cuda = USE_CUDA
args.seed = SEED
args.N_E, args.N_CV, args.N_T = N_E, N_CV, N_T
args.T, args.T_test = T, T

def make_filter_model(evolution, Q, R):
    """The SystemModel handed to the *filters*: same noises as the data, but `evolution` is either
    the true f (`full`) or f_partial (`partial`), and P_0 is the non-degenerate FILTER_P0 * I.
    """
    sys_model = SystemModel(evolution, Q, h, R, T, T, m, n)
    sys_model.InitSequence(m1x_0, FILTER_P0 * torch.eye(m))
    return sys_model

def make_dataset(db):
    """Generates (or loads) the dataset for noise level `db` and builds its per-scenario filter models."""
    r2 = 10 ** (-db / 10.0)                 # 1/R^2 [dB] = -10 log10(r2)
    q2 = 10 ** (NU_DB / 10.0) * r2          # nu = q2/r2 held fixed
    Q, R = q2 * Q_structure, r2 * R_structure

    path = os.path.join(DATA_DIR, f"data_R{db:+g}dB.pt")
    if REGEN_DATA or not os.path.exists(path):
        sys_true = SystemModel(f, Q, h, R, T, T, m, n)   # the data comes from the TRUE model
        sys_true.InitSequence(m1x_0, m2x_0)
        torch.manual_seed(SEED); np.random.seed(SEED); random.seed(SEED)   # same data on every re-run
        DataGen(args, sys_true, path)

    # DataGen saves a plain list of tensors (splits first, then the initial conditions).
    train_y, train_x, cv_y, cv_x, test_y, test_x = torch.load(
        path, map_location=DEVICE, weights_only=False)[:6]

    return {"train_y": train_y, "train_x": train_x,
            "cv_y": cv_y, "cv_x": cv_x,
            "test_y": test_y, "test_x": test_x,
            "r2": r2, "q2": q2,
            # one filter SystemModel per scenario: identical data, different dynamics
            "sys": {s: make_filter_model(f if s == "full" else f_partial, Q, R) for s in SCENARIOS}}

datasets = {db: make_dataset(db) for db in tqdm(INV_R2_DB, desc="datasets", unit="level")}

# Setup Summary
print(f"{len(datasets)} dataset(s) ready in {DATA_DIR}/   (scenarios: {', '.join(SCENARIOS)})\n")
print(f"{'1/R^2 [dB]':>10} {'r2':>10} {'q2':>10}   train / cv / test   shapes (y | x)")
for db, ds in datasets.items():
    print(f"{db:>10.4g} {ds['r2']:>10.4g} {ds['q2']:>10.4g}   "
          f"{len(ds['train_y']):5d} / {len(ds['cv_y']):3d} / {len(ds['test_y']):3d}   "
          f"{tuple(ds['train_y'].shape[1:])} | {tuple(ds['train_x'].shape[1:])}")
print(f"\ntarget alignment: input and target both carry T = {T} columns, as Pipeline_REKF requires")

datasets:   0%|          | 0/4 [00:00<?, ?level/s]

4 dataset(s) ready in Results_task4\datasets/   (scenarios: full, partial)

1/R^2 [dB]         r2         q2   train / cv / test   shapes (y | x)
    -12.04         16       0.16     120 /  30 /  30   (2, 100) | (2, 100)
     -6.02      3.999    0.03999     120 /  30 /  30   (2, 100) | (2, 100)
         0          1       0.01     120 /  30 /  30   (2, 100) | (2, 100)
        10        0.1      0.001     120 /  30 /  30   (2, 100) | (2, 100)

target alignment: input and target both carry T = 100 columns, as Pipeline_REKF requires


## 4. Data Visualization

In [5]:
# Reference trajectory: one randomly-chosen training sequence at the first noise level.
db_show = INV_R2_DB[3]
ds_show = datasets[db_show]
k_show = random.randrange(len(ds_show["train_y"]))

x_show = ds_show["train_x"][k_show]                  # [m, T] latent state
y_show = ds_show["train_y"][k_show]                  # [n, T] noisy observation
y_clean = h(x_show)                                  # [n, T] noiseless h(x_t)

fig = make_subplots(rows=4, cols=1, shared_xaxes=True, vertical_spacing=0.05)
fig.add_trace(go.Scatter(y=x_show[0].cpu().numpy(), mode="lines", showlegend=False,
                         line=dict(color="#1f77b4", width=1.2)), row=1, col=1)
fig.add_trace(go.Scatter(y=x_show[1].cpu().numpy(), mode="lines", showlegend=False,
                         line=dict(color="#d62728", width=1.2)), row=2, col=1)
for row, comp in ((3, 0), (4, 1)):        # one panel per observation channel
    fig.add_trace(go.Scatter(y=y_clean[comp].cpu().numpy(), mode="lines", name="h(x) (noiseless)",
                             legendgroup="clean", showlegend=(row == 3),
                             line=dict(color="black", width=1.8)), row=row, col=1)
    fig.add_trace(go.Scatter(y=y_show[comp].cpu().numpy(), mode="lines", name="y (measured)",
                             legendgroup="meas", showlegend=(row == 3),
                             line=dict(color="#2ca02c", width=1.0)), row=row, col=1)

fig.update_yaxes(title_text="x₁", row=1, col=1)
fig.update_yaxes(title_text="x₂", row=2, col=1)
fig.update_yaxes(title_text="y₁", row=3, col=1)
fig.update_yaxes(title_text="y₂", row=4, col=1)
fig.update_xaxes(title_text="t", row=4, col=1)
fig.update_layout(title=f"Training trajectory #{k_show} at 1/R² = {db_show:g} dB",
                  height=820, width=750, template="plotly_white", hovermode="x unified")
fig.show()

# The same panels across the sweep: how much of y is signal and how much is noise at each level.
if len(INV_R2_DB) > 1:
    fig = make_subplots(rows=2, cols=len(INV_R2_DB), shared_xaxes=True, vertical_spacing=0.07,
                        subplot_titles=[f"1/R² = {db:g} dB" for db in INV_R2_DB])
    for col, db in enumerate(INV_R2_DB, start=1):
        ds = datasets[db]
        y_lvl, y_lvl_clean = ds["train_y"][k_show], h(ds["train_x"][k_show])
        for row, comp in ((1, 0), (2, 1)):
            first = (col == 1 and row == 1)       # the legend is drawn once for the whole figure
            fig.add_trace(go.Scatter(y=y_lvl_clean[comp].cpu().numpy(), mode="lines",
                                     name="h(x) (noiseless)", legendgroup="clean", showlegend=first,
                                     line=dict(color="black", width=1.8)), row=row, col=col)
            fig.add_trace(go.Scatter(y=y_lvl[comp].cpu().numpy(), mode="lines",
                                     name="y (measured)", legendgroup="meas", showlegend=first,
                                     line=dict(color="#2ca02c", width=1.0)), row=row, col=col)
        fig.update_xaxes(title_text="t", row=2, col=col)
    fig.update_yaxes(title_text="y₁", row=1, col=1)
    fig.update_yaxes(title_text="y₂", row=2, col=1)
    fig.update_layout(title=f"Observation of training trajectory #{k_show} across the noise sweep",
                      height=620, width=380 * len(INV_R2_DB), template="plotly_white")
    fig.show()

## 5. Estimators Instantiation

In [6]:
# KalmanNet reads its layer widths off `args` inside NNBuild, so that is where KNET_MODEL belongs.
for key, value in KNET_MODEL.items():
    setattr(args, key, value)

def align_prior(model, sysmdl):
    """Shifts the REKF family's initial prior by one dynamics step.

    `sysmdl.f` — not the true f — so the partial-information filter stays blind to
    the real dynamics.
    """
    model.x0 = torch.transpose(sysmdl.f(sysmdl.m1x_0), 0, 1)   # [1, m], read by reset_state
    return model

def build_rekf(sysmdl, y_seq):
    """Analytic robust EKF: the RT-KalmanNet filter class with its network switched off.
    The tolerance passed here is a placeholder — the grid search overwrites `model.c`.
    """
    return align_prior(Prop_RTKnet(sysmdl, y_seq, c=1e-3, use_nn=False, sl_model=0), sysmdl)

def build_knet(sysmdl, y_seq):
    """KalmanNet: a plain nn.Module driven by Pipeline_EKF (`y_seq` unused, see above).
    Predicts before correcting, so its prior needs no `align_prior` shift.
    """
    torch.manual_seed(SEED)
    net = KalmanNetNN()
    net.NNBuild(sysmdl, args)
    return net

def build_original(sysmdl, y_seq):
    """Original RT-KalmanNet: MLP with output feedback, estimating the tolerance online."""
    torch.manual_seed(SEED)
    return align_prior(Orig_RTKnet(sysmdl, y_seq, use_nn=True, sl_model=0, **ORIG_MODEL), sysmdl)

def build_proposed(sysmdl, y_seq):
    """Proposed RT-KalmanNet: GRU estimating the tolerance online."""
    torch.manual_seed(SEED)
    return align_prior(Prop_RTKnet(sysmdl, y_seq, use_nn=True, sl_model=0, **PROP_MODEL), sysmdl)

# Canonical order, used by every table and figure downstream. The EKF has no entry in BUILD: it is a
# function call, not an object.
METHODS = [name for name, enabled in (("EKF", INCLUDE_EKF), ("REKF", INCLUDE_REKF),
                                      ("KalmanNet", INCLUDE_KNET), ("Original", INCLUDE_ORIGINAL),
                                      ("Proposed", INCLUDE_PROPOSED)) if enabled]
BUILD = {"REKF": build_rekf, "KalmanNet": build_knet,
         "Original": build_original, "Proposed": build_proposed}
LEARNED = [name for name in METHODS if name in ("KalmanNet", "Original", "Proposed")]

## 6. REKF Baseline — Tolerance grid search

The analytic Robust EKF uses **one constant** tolerance $c$. Following [UAVF] Sec. IV, $c^*$ is the
grid value minimising the state MSE, and it is selected **independently for every (scenario, noise
level)**.

In [7]:
def run_filter(model, y, train=False):
    """Runs one observation sequence through a filter of the RT-KalmanNet family.

    fnREKF returns a list whose last element is the elapsed time and whose element [1] is the
    estimated tolerance sequence (None for the analytic REKF). Shared with the test and timing
    sections below, which is why it returns more than the grid search needs.
    """
    model.reset_state(y)
    out = model.fnREKF(train=train)
    c_arr = out[1]
    c_traj = None if c_arr is None else np.asarray([float(c) for c in c_arr], dtype=np.float64)
    return model.Xn, c_traj, float(out[-1])

c_star = {}          # (scenario, inv_r2_db) -> best constant tolerance
if INCLUDE_REKF:
    # One scoring subset, drawn once from the global seed: every candidate c -- and every
    # (scenario, noise level) -- is then scored on the very same training trajectories.
    random.seed(SEED)
    c_idx = random.sample(range(N_E), min(N_C_SEQ, N_E))

    grid_rows = []
    for db in INV_R2_DB:
        ds = datasets[db]
        for scen in SCENARIOS:
            rekf = build_rekf(ds["sys"][scen], ds["train_y"][0])
            grid = {}
            with torch.no_grad():
                for cand in tqdm(C_GRID, desc=f"REKF grid [{scen} | {db:+g} dB]",
                                 unit="c", leave=False):
                    cand = float(cand)
                    # With use_nn=False the tolerance is a fixed tensor read by fnComputeTheta.
                    rekf.c = torch.tensor(cand, device=DEVICE, dtype=torch.float32)
                    losses = [crit(run_filter(rekf, ds["train_y"][i])[0], ds["train_x"][i]).item()
                              for i in c_idx]
                    grid[cand] = float(np.mean(losses))
            best = min(grid, key=grid.get)
            c_star[(scen, db)] = best
            grid_rows.append((scen, db, best, to_dB(grid[best])))

    print(f"c* minimises the state MSE over {len(C_GRID)} candidates in "
          f"[{C_GRID[0]:.4g}, {C_GRID[-1]:.4g}], scored on training sequences {c_idx}")
    print(f"{'scenario':>9} {'1/R^2 [dB]':>11} {'c*':>10} {'train MSE [dB]':>15}")
    for scen, db, best, mse_db in grid_rows:
        print(f"{scen:>9} {db:>11.4g} {best:>10.4g} {mse_db:>15.3f}")
else:
    print("INCLUDE_REKF = False: no tolerance grid search")


REKF grid [full | -12.04 dB]:   0%|          | 0/20 [00:00<?, ?c/s]

REKF grid [partial | -12.04 dB]:   0%|          | 0/20 [00:00<?, ?c/s]

REKF grid [full | -6.02 dB]:   0%|          | 0/20 [00:00<?, ?c/s]

REKF grid [partial | -6.02 dB]:   0%|          | 0/20 [00:00<?, ?c/s]

REKF grid [full | +0 dB]:   0%|          | 0/20 [00:00<?, ?c/s]

REKF grid [partial | +0 dB]:   0%|          | 0/20 [00:00<?, ?c/s]

REKF grid [full | +10 dB]:   0%|          | 0/20 [00:00<?, ?c/s]

REKF grid [partial | +10 dB]:   0%|          | 0/20 [00:00<?, ?c/s]

c* minimises the state MSE over 20 candidates in [0.0001, 1], scored on training sequences [108, 49, 97, 113, 53, 5]
 scenario  1/R^2 [dB]         c*  train MSE [dB]
     full      -12.04     0.1438          -5.977
  partial      -12.04  0.0001624          -2.288
     full       -6.02    0.05456         -13.407
  partial       -6.02    0.02069          -7.101
     full           0    0.02069         -19.723
  partial           0     0.2336         -10.703
     full          10    0.01274         -29.814
  partial          10          1         -14.503


## 7. Training the Learned Estimators Across the Sweep

Every learned model is trained **from scratch for each (scenario, noise level)**. 
Total: `len(INV_R2_DB) x len(SCENARIOS) x len(LEARNED)` trainings.

> **Note on the Original network.** Its tolerance reaches the loss only through the `theta`
> bisection, a sequence of branches on `c`, and its output feedback is detached — so its parameters
> receive no gradient and "training" reduces to picking the best-validation epoch of an essentially
> unchanged network. This is a structural property of the original design and exactly the
> comparison being made, not a bug; the flat curves in Sec. 8 are its expected signature.

In [8]:
def r_tag(db):
    """Filename-safe tag for a noise level: 10 -> 'R+10dB', -12.04 -> 'R-12.04dB'."""
    return f"R{db:+g}dB"

def ckpt_path(method, scen, db):
    """Unique checkpoint path for one combination."""
    return os.path.join(CKPT_DIR, f"{method}_{scen}_{r_tag(db)}.pt")

def train_args(params):
    """A fresh `args` for one training: config.py's defaults, then `params`."""
    a = config.general_settings()
    a.use_cuda, a.seed = USE_CUDA, SEED
    for key, value in params.items():
        setattr(a, key, value)
    return a

TRAIN = {"KalmanNet": KNET_TRAIN, "Original": ORIG_TRAIN, "Proposed": PROP_TRAIN}
history, train_times = {}, {}          # both keyed (method, scenario, inv_r2_db)

def train_one(method, scen, db):
    """Trains `method` from scratch on one (scenario, noise level), then records its loss curves,
    its wall-clock time and its checkpoint.
    """
    ds = datasets[db]
    sysmdl, y0 = ds["sys"][scen], ds["train_y"][0]
    args_train = train_args(TRAIN[method])
    tag = f"{method}_{scen}_{r_tag(db)}"

    # `passes` = sequence-passes implied by the configuration. n_steps means gradient steps on one
    # sampled mini-batch for Pipeline_EKF but epochs (a full sweep of the training set) for
    # Pipeline_REKF, so the elapsed times below are only readable next to this count.
    if method == "KalmanNet":
        pipeline = Pipeline_EKF("", CKPT_DIR, tag)
        pipeline.setModel(build_knet(sysmdl, y0))
        written = PATH_RESULTS + "best-model.pt"
        passes = args_train.n_steps * args_train.n_batch
    else:
        pipeline = Pipeline_REKF(CKPT_DIR, tag, allow_tbptt=(method == "Proposed"))
        pipeline.setModel(BUILD[method](sysmdl, y0))
        written = pipeline._best_model_path(PATH_RESULTS)
        passes = args_train.n_steps * N_E
    pipeline.setssModel(sysmdl)
    pipeline.setTrainingParams(args_train)

    print(f"\n##### {method} | {scen} information | 1/R^2 = {db:+g} dB #####")
    t0 = time.time()
    _, cv_dB, _, tr_dB = pipeline.NNTrain(sysmdl, ds["cv_y"], ds["cv_x"],
                                          ds["train_y"], ds["train_x"], PATH_RESULTS)
    elapsed = time.time() - t0

    history[(method, scen, db)] = {"train_dB": tr_dB.tolist(), "val_dB": cv_dB.tolist()}
    train_times[(method, scen, db)] = elapsed
    dst = ckpt_path(method, scen, db)
    os.replace(written, dst)           # claim the fixed name before the next training reuses it
    print(f"  best CV = {float(pipeline.MSE_cv_dB_opt):+.4f} dB "
          f"(epoch {pipeline.MSE_cv_idx_opt + 1}) | {elapsed:.0f}s for {passes} sequence-passes "
          f"| checkpoint: {dst}")

combos = [(db, scen) for db in INV_R2_DB for scen in SCENARIOS]

# `history` / `train_times` are keyed by the (method, scenario, noise) tuple, which JSON cannot
# store, so the key is flattened on the way out and rebuilt on the way in.
def _hkey(method, scen, db):
    return f"{method}|{scen}|{db:+g}"

def _hparse(key):
    method, scen, db = key.split("|")
    return (method, scen, float(db))

if RETRAIN:
    for method in LEARNED:
        for db, scen in tqdm(combos, desc=f"training {method}", unit="combo"):
            train_one(method, scen, db)
    with open(HIST_PATH, "w") as fh:
        json.dump({"history":     {_hkey(*k): v for k, v in history.items()},
                   "train_times": {_hkey(*k): v for k, v in train_times.items()}}, fh, indent=1)
    print(f"\nsaved training + validation history for {len(history)} combination(s) -> {HIST_PATH}")

elif os.path.exists(HIST_PATH):
    with open(HIST_PATH) as fh:
        _saved = json.load(fh)
    history.update({_hparse(k): v for k, v in _saved["history"].items()})
    train_times.update({_hparse(k): v for k, v in _saved["train_times"].items()})
    print(f"RETRAIN = False: nothing trained; loaded history for {len(history)} "
          f"combination(s) <- {HIST_PATH}")

else:
    print(f"RETRAIN = False but no history file at {HIST_PATH}: Sec. 9 still reuses the "
          f"checkpoints, but Sec. 8 has no loss curves to draw")

RETRAIN = False: nothing trained; loaded history for 24 combination(s) <- Results_task4\training_history.json


## 8. Training and Validation Loss

In [9]:
# One panel per available case: rows are (method, scenario), columns the noise levels -- the whole
# campaign, since every combination is trained anyway. Reads `history`, which
# Sec. 7 either fills by training or loads from training_history.json, so this works with
# RETRAIN = False.
panel_rows = [(method, scen) for method in LEARNED for scen in SCENARIOS
              if any((method, scen, db) in history for db in INV_R2_DB)]

if not panel_rows:
    print("no training history available "
          "(Sec. 7 skipped, no saved training_history.json, or no learned estimator enabled)")
else:
    fig = make_subplots(rows=len(panel_rows), cols=len(INV_R2_DB), shared_xaxes=False,
                        vertical_spacing=min(0.08, 0.9 / max(1, len(panel_rows))),
                        subplot_titles=[f"{method} [{scen}] | {db:+g} dB"
                                        for method, scen in panel_rows for db in INV_R2_DB])
    for row, (method, scen) in enumerate(panel_rows, start=1):
        n_pts = 0
        for col, db in enumerate(INV_R2_DB, start=1):
            hist = history.get((method, scen, db))
            if hist is None:
                continue
            ep = list(range(1, len(hist["train_dB"]) + 1))
            n_pts = max(n_pts, len(ep))
            first = (row == 1 and col == 1)      # the legend is drawn once for the whole figure
            fig.add_trace(go.Scatter(x=ep, y=hist["train_dB"], mode="lines", name="training",
                                     legendgroup="train", showlegend=first,
                                     line=dict(color="#1f77b4", width=2)), row=row, col=col)
            fig.add_trace(go.Scatter(x=ep, y=hist["val_dB"], mode="lines", name="validation",
                                     legendgroup="val", showlegend=first,
                                     line=dict(color="#ff7f0e", width=2, dash="dash")),
                          row=row, col=col)
        # ~10 ticks whatever the row's length. Every panel in a row runs the same number of epochs
        # (same TRAIN[method]), so one dtick from the row's longest curve fits all its columns.
        unit = "gradient step" if method == "KalmanNet" else "epoch"
        for col in range(1, len(INV_R2_DB) + 1):
            fig.update_xaxes(title_text=unit, dtick=max(1, round(n_pts / 10)), row=row, col=col)
        fig.update_yaxes(title_text="MSE [dB]", row=row, col=1)
    fig.update_layout(title="Training and validation loss — all cases",
                      height=300 * len(panel_rows), width=340 * len(INV_R2_DB) + 120,
                      template="plotly_white", hovermode="x unified")
    fig.show()

## 9. Test — State MSE Across the Full Sweep

Every estimator is evaluated on the **same** test trajectories of each dataset.

The EKF makes its only appearance here: it is analytic, needs neither tuning nor training, and
serves as the reference floor for the full-information scenario and as the reference *failure*
under model mismatch.

In [10]:
def mse_stats_db(v):
    """(mean_dB, spread_dB) from a vector of per-sequence LINEAR MSEs.

    spread_dB follows the project convention 10log10(std + mean) - mean_dB.
    """
    v = np.asarray(v, dtype=np.float64)
    mean_db = to_dB(v.mean())
    spread_db = to_dB(v.std(ddof=1) + v.mean()) - mean_db if v.size > 1 else 0.0
    return mean_db, spread_db

def test_ekf(ds, scen):
    """Test the analytic EKF."""
    with contextlib.redirect_stdout(io.StringIO()):
        out = EKFTest(args, ds["sys"][scen], ds["test_y"], ds["test_x"])
    return np.asarray(out[0], dtype=np.float64)

def test_rekf(ds, scen, db):
    """Analytic robust EKF at the constant tolerance selected for this combination."""
    rekf = build_rekf(ds["sys"][scen], ds["test_y"][0])
    rekf.c = torch.tensor(c_star[(scen, db)], device=DEVICE, dtype=torch.float32)
    with torch.no_grad():
        return np.asarray([crit(run_filter(rekf, y)[0], x).item()
                           for y, x in zip(ds["test_y"], ds["test_x"])], dtype=np.float64)

def make_pipeline(method, ds, scen, db):
    """The pipeline owning `method`, ready to test the (scenario, noise) checkpoint."""
    ckpt = ckpt_path(method, scen, db)
    if not os.path.exists(ckpt):
        return None, None
    sysmdl, y0 = ds["sys"][scen], ds["train_y"][0]
    tag = f"{method}_{scen}_{r_tag(db)}"
    pipeline = (Pipeline_EKF("", CKPT_DIR, tag) if method == "KalmanNet"
                else Pipeline_REKF(CKPT_DIR, tag, allow_tbptt=(method == "Proposed")))
    pipeline.setModel(BUILD[method](sysmdl, y0))   # placeholder: NNTest loads the checkpoint into it
    pipeline.setssModel(sysmdl)
    pipeline.setTrainingParams(train_args(TRAIN[method]))
    return pipeline, ckpt

def test_learned(pipeline, method, sysmdl, y, x, ckpt):
    """One test pass through the owning pipeline, returning that pipeline's own list."""
    if method == "KalmanNet":
        return pipeline.NNTest(sysmdl, y, x, PATH_RESULTS, load_model=True, load_model_path=ckpt)
    return pipeline.NNTest(sysmdl, y, x, PATH_RESULTS, ckpt_path=ckpt)

records, per_seq_rows = [], []
for db in INV_R2_DB:
    ds = datasets[db]
    for scen in SCENARIOS:
        print(f"\n--- {scen} information | 1/R^2 = {db:+g} dB ---")
        for method in METHODS:
            if method == "EKF":
                mse = test_ekf(ds, scen)
            elif method == "REKF":
                mse = test_rekf(ds, scen, db)
            else:
                pipeline, ckpt = make_pipeline(method, ds, scen, db)
                if pipeline is None:
                    print(f"  {method:10s}: {ckpt_path(method, scen, db)} missing -- skipped")
                    continue
                mse = test_learned(pipeline, method, ds["sys"][scen],
                                   ds["test_y"], ds["test_x"], ckpt)[0]

            mean_db, std_db = mse_stats_db(mse)
            records.append({"model": method, "scenario": scen, "inv_r2_db": db,
                            "mse_db": round(mean_db, 4), "std_db": round(std_db, 4)})
            per_seq_rows += [{"model": method, "scenario": scen, "inv_r2_db": db,
                              "sequence": k, "mse": float(v)} for k, v in enumerate(mse)]
            print(f"  {method:10s}: MSE = {mean_db:+8.3f} dB  (+/-{abs(std_db):.2f})")

results_df = pd.DataFrame(records)
per_seq_df = pd.DataFrame(per_seq_rows)
results_df.to_csv(os.path.join(CKPT_DIR, "results_long.csv"), index=False)
per_seq_df.to_csv(os.path.join(CKPT_DIR, "per_sequence_metrics.csv"), index=False)

print(f"\n{len(results_df)} results over {N_T} test trajectories (T = {T}) -> "
      f"{CKPT_DIR}/results_long.csv and {CKPT_DIR}/per_sequence_metrics.csv")
results_df


--- full information | 1/R^2 = -12.04 dB ---
  EKF       : MSE =   -6.065 dB  (+/-0.85)
  REKF      : MSE =   -6.052 dB  (+/-0.85)
  KalmanNet : MSE =   -6.337 dB  (+/-0.78)
  Original  : MSE =   -5.645 dB  (+/-1.07)
  Proposed  : MSE =   -6.061 dB  (+/-0.85)

--- partial information | 1/R^2 = -12.04 dB ---
  EKF       : MSE =   -2.193 dB  (+/-1.43)
  REKF      : MSE =   -1.852 dB  (+/-1.35)
  KalmanNet : MSE =   -6.319 dB  (+/-0.76)
  Original  : MSE =   +2.452 dB  (+/-1.24)
  Proposed  : MSE =   -1.593 dB  (+/-1.26)

--- full information | 1/R^2 = -6.02 dB ---
  EKF       : MSE =  -13.268 dB  (+/-0.57)
  REKF      : MSE =  -13.256 dB  (+/-0.57)
  KalmanNet : MSE =  -13.300 dB  (+/-0.58)
  Original  : MSE =  -13.135 dB  (+/-0.56)
  Proposed  : MSE =  -13.259 dB  (+/-0.57)

--- partial information | 1/R^2 = -6.02 dB ---
  EKF       : MSE =   -5.749 dB  (+/-2.08)
  REKF      : MSE =   -4.133 dB  (+/-2.48)
  KalmanNet : MSE =  -12.546 dB  (+/-0.55)
  Original  : MSE =   -0.176 dB  (+/-1

,model,scenario,inv_r2_db,mse_db,std_db
0,EKF,full,-12.04,-6.0654,0.8532
1,REKF,full,-12.04,-6.0517,0.8454
2,KalmanNet,full,-12.04,-6.3369,0.7834
3,Original,full,-12.04,-5.6449,1.0702
4,Proposed,full,-12.04,-6.0613,0.8515
5,EKF,partial,-12.04,-2.1931,1.4315
6,REKF,partial,-12.04,-1.8521,1.3541
7,KalmanNet,partial,-12.04,-6.3194,0.7603
8,Original,partial,-12.04,2.4525,1.2383
9,Proposed,partial,-12.04,-1.5927,1.2618


## 10. Headline Figure — MSE vs 1/R²

The main result of Task 4: posterior MSE [dB] against measurement noise 1/R² [dB], **one panel per
scenario**, one curve per estimator. Both panels share the same y-range, because the whole point of
the comparison is how much the *partial*-information penalty costs each method relative to the same
absolute scale.

In [11]:
MARKERS = {"EKF": "circle", "REKF": "square", "KalmanNet": "triangle-up",
           "Original": "diamond", "Proposed": "star"}
COLORS = {"EKF": "#2ca02c", "REKF": "#7f7f7f", "KalmanNet": "#d62728",
          "Original": "#ff7f0e", "Proposed": "#1f77b4"}

if results_df.empty:
    print("Sec. 9 produced no records: nothing to draw")
else:
    fig = make_subplots(rows=1, cols=len(SCENARIOS), shared_yaxes=True, horizontal_spacing=0.05,
                        subplot_titles=[f"{scen.capitalize()} information" for scen in SCENARIOS])
    for col, scen in enumerate(SCENARIOS, start=1):
        for method in METHODS:
            d = results_df[(results_df["scenario"] == scen)
                           & (results_df["model"] == method)].sort_values("inv_r2_db")
            if d.empty:
                continue
            fig.add_trace(go.Scatter(
                x=d["inv_r2_db"], y=d["mse_db"], mode="lines+markers", name=method,
                legendgroup=method, showlegend=(col == 1),      # one legend for the whole figure
                line=dict(color=COLORS[method], width=1.8),
                marker=dict(symbol=MARKERS[method], size=10),
                error_y=dict(type="data", array=d["std_db"].abs(), visible=True,
                             thickness=1.2, width=4)), row=1, col=col)
        fig.update_xaxes(title_text="1/R² [dB]", showgrid=True, row=1, col=col)

    # One absolute scale for both panels: the question is what the partial-information penalty
    # costs each method, which auto-scaled panels would hide.
    lo = (results_df["mse_db"] - results_df["std_db"].abs()).min()
    hi = (results_df["mse_db"] + results_df["std_db"].abs()).max()
    pad = 0.05 * max(hi - lo, 1.0)
    fig.update_yaxes(range=[lo - pad, hi + pad], showgrid=True)
    fig.update_yaxes(title_text="MSE [dB]", row=1, col=1)
    fig.update_layout(title="Posterior state MSE vs measurement noise",
                      height=470, width=520 * len(SCENARIOS), template="plotly_white",
                      hovermode="x unified")
    fig.show()

## 11. Inference-Time Benchmark

MSE alone does not decide which estimator is deployable: the REKF family pays a **theta bisection at
every time step**, and the question is what that costs relative to KalmanNet, which has no bisection
at all. Measured at **every** noise level of the sweep, one row of panels per level: the recursion
cost itself depends on `T` and on the architecture rather than on the noise, so the rows also show
how stable that cost is — while the accuracy axis on the right does move with the noise.

In [12]:
random.seed(SEED)                      # the same sequences are timed for every method and level
time_idx = random.sample(range(N_T), min(N_TIME_SEQ, N_T))

timing_rows = []
for db in INV_R2_DB:
    ds = datasets[db]
    for scen in SCENARIOS:
        sysmdl = ds["sys"][scen]
        for method in METHODS:
            if method in LEARNED:
                pipeline, ckpt = make_pipeline(method, ds, scen, db)
                if pipeline is None:
                    print(f"{method:10s} [{scen}]: no checkpoint at {db:+g} dB -- skipped")
                    continue
            elif method == "REKF":
                rekf = build_rekf(sysmdl, ds["test_y"][0])
                rekf.c = torch.tensor(c_star[(scen, db)], device=DEVICE, dtype=torch.float32)

            times = []
            # EKFTest and Pipeline_EKF.NNTest also compute the std of the per-sequence MSE across
            # the batch; on a 1-sequence slice that is undefined -- and unused here, since only
            # comp_time is read -- so torch's degrees-of-freedom warning is silenced.
            with torch.no_grad(), warnings.catch_warnings():
                warnings.filterwarnings("ignore", message=r"std\(\): degrees of freedom")
                for k in tqdm(time_idx, desc=f"timing {method} [{scen} | {db:+g} dB]",
                              unit="traj", leave=False):
                    y, x = ds["test_y"][k:k + 1], ds["test_x"][k:k + 1]   # 1-sequence slice
                    if method == "EKF":
                        t0 = time.time()
                        with contextlib.redirect_stdout(io.StringIO()):   # see test_ekf
                            EKFTest(args, sysmdl, y, x)
                        times.append(time.time() - t0)
                    elif method == "REKF":
                        times.append(run_filter(rekf, y[0])[2])
                    else:
                        times.append(test_learned(pipeline, method, sysmdl, y, x, ckpt)[-1])

            times = np.asarray(times, dtype=np.float64)
            # Sec. 9 skipped exactly the combinations skipped above, so this lookup always hits.
            mse_db = results_df[(results_df["model"] == method) & (results_df["scenario"] == scen)
                                & (results_df["inv_r2_db"] == db)]["mse_db"].iloc[0]
            timing_rows.append({
                "model": method, "scenario": scen, "inv_r2_db": db,
                "time_per_traj_s": round(float(times.mean()), 4),
                "time_std_s": round(float(times.std(ddof=1)) if times.size > 1 else 0.0, 4),
                "time_per_step_ms": round(1e3 * float(times.mean()) / T, 3),
                "train_time_s": round(train_times.get((method, scen, db), float("nan")), 1),
                "mse_db": float(mse_db),
            })

timing_df = pd.DataFrame(timing_rows)
timing_df.to_csv(os.path.join(CKPT_DIR, "timing_focus.csv"), index=False)
print(timing_df.to_string(index=False))

if timing_df.empty:
    print("no timing measurements to draw")
else:
    # The same pair of panels repeated one row per noise level, stacked so the rows can be read
    # against each other. Only the levels that produced measurements get a row.
    timing_dbs = [db for db in INV_R2_DB if (timing_df["inv_r2_db"] == db).any()]
    fig = make_subplots(rows=len(timing_dbs), cols=2,
                        horizontal_spacing=0.12, vertical_spacing=0.10,
                        subplot_titles=[title for db in timing_dbs for title in
                                        (f"Inference time per trajectory | 1/R² = {db:+g} dB",
                                         f"Accuracy vs cost | 1/R² = {db:+g} dB")])

    for row, db in enumerate(timing_dbs, start=1):
        at_db = timing_df[timing_df["inv_r2_db"] == db]
        first = (row == 1)                    # the legend is drawn once for the whole figure

        # left: one bar group per method, one trace per scenario (hatched = partial information)
        for scen in SCENARIOS:
            d = at_db[at_db["scenario"] == scen]
            fig.add_trace(go.Bar(x=d["model"], y=d["time_per_traj_s"], name=scen,
                                 legendgroup=scen, showlegend=first,
                                 error_y=dict(type="data", array=d["time_std_s"], visible=True),
                                 marker=dict(color=[COLORS[m] for m in d["model"]],
                                             pattern_shape="" if scen == "full" else "/")),
                          row=row, col=1)

        # right: the same numbers as a trade-off -- MSE against the scale-free per-step cost. One
        # dot per (method, scenario), hollow for partial information; the text label carries the
        # identity and the hover names both coordinates as the averages they are.
        for rec in at_db.to_dict("records"):
            fig.add_trace(go.Scatter(
                x=[rec["time_per_step_ms"]], y=[rec["mse_db"]], mode="markers+text",
                text=[f"{rec['model']} [{rec['scenario'][:4]}]"], textposition="top center",
                textfont=dict(size=9), showlegend=False,
                marker=dict(symbol="circle" if rec["scenario"] == "full" else "circle-open",
                            size=11, color=COLORS[rec["model"]], line=dict(width=2)),
                hovertemplate=(f"<b>{rec['model']} [{rec['scenario']}]</b><br>"
                               f"mean time = %{{x:.3f}} ms/step ({len(time_idx)} sequences)<br>"
                               f"MSE = %{{y:.2f}} dB (mean over the test set)<extra></extra>")),
                row=row, col=2)

        fig.update_yaxes(title_text="time per trajectory [s]", type="log", row=row, col=1)
        fig.update_xaxes(title_text="time per step [ms]", type="log", row=row, col=2)
        fig.update_yaxes(title_text="MSE [dB]", row=row, col=2)

    fig.update_layout(title="Inference cost across the noise sweep", barmode="group",
                      height=430 * len(timing_dbs), width=1200, template="plotly_white")
    fig.show()

timing EKF [full | -12.04 dB]:   0%|          | 0/10 [00:00<?, ?traj/s]

timing REKF [full | -12.04 dB]:   0%|          | 0/10 [00:00<?, ?traj/s]

timing KalmanNet [full | -12.04 dB]:   0%|          | 0/10 [00:00<?, ?traj/s]

timing Original [full | -12.04 dB]:   0%|          | 0/10 [00:00<?, ?traj/s]

timing Proposed [full | -12.04 dB]:   0%|          | 0/10 [00:00<?, ?traj/s]

...

timing REKF [partial | +10 dB]:   0%|          | 0/10 [00:00<?, ?traj/s]

timing KalmanNet [partial | +10 dB]:   0%|          | 0/10 [00:00<?, ?traj/s]

timing Original [partial | +10 dB]:   0%|          | 0/10 [00:00<?, ?traj/s]

timing Proposed [partial | +10 dB]:   0%|          | 0/10 [00:00<?, ?traj/s]

    model scenario  inv_r2_db  time_per_traj_s  time_std_s  time_per_step_ms  train_time_s   mse_db
      EKF     full     -12.04           0.1436      0.0079             1.436           NaN  -6.0654
     REKF     full     -12.04           0.5447      0.0213             5.447           NaN  -6.0517
KalmanNet     full     -12.04           0.1301      0.0094             1.301           9.4  -6.3369
 Original     full     -12.04           0.6710      0.0070             6.710        1144.8  -5.6449
 Proposed     full     -12.04           0.7200      0.0099             7.200        2859.3  -6.0613
      EKF  partial     -12.04           0.1285      0.0155             1.285           NaN  -2.1931
     REKF  partial     -12.04           0.4143      0.0205             4.143           NaN  -1.8521
KalmanNet  partial     -12.04           0.1301      0.0106             1.301           9.1  -6.3194
 Original  partial     -12.04           0.6547      0.0115             6.547        1129.9   2.4525


## 12. Estimated tolerance $\hat c_t$ across the eight experimental cases

One subplot per case (rows are the information scenario, columns the noise level) so that the curves stay separable. Each panel holds the REKF's constant $c^\star$ (dashed) and the tolerance estimated online by the original and proposed RT-KalmanNet, averaged over the first `N_C_PLOT` test trajectories. KalmanNet and the EKF are absent because neither carries a tolerance. The estimates are produced from the saved checkpoints, so this section works with `RETRAIN = False`.

In [13]:
# Estimated tolerance per experimental case. Reads the checkpoints written by Sec. 7,
# so it needs no training. REKF contributes the constant c* selected in Sec. 6.
N_C_PLOT = min(10, N_T)          # test trajectories averaged in each panel

def c_trajectories(method, ds, scen, db, n_seq):
    """Estimated tolerance for `n_seq` test trajectories, taken from the saved checkpoint."""
    ckpt = ckpt_path(method, scen, db)
    if not os.path.exists(ckpt):
        return None
    model = BUILD[method](ds["sys"][scen], ds["train_y"][0])
    model.nn.load_state_dict(torch.load(ckpt, map_location=DEVICE, weights_only=True))
    model.nn.eval()
    trajs = []
    with torch.no_grad():
        for k in range(min(n_seq, len(ds["test_y"]))):
            c_traj = run_filter(model, ds["test_y"][k])[1]
            if c_traj is not None:
                trajs.append(c_traj)
    return np.stack(trajs) if trajs else None      # [n_seq, T]

C_METHODS = [m for m in ("Original", "Proposed") if m in METHODS]
c_curves, c_rows = {}, []
for db in tqdm(INV_R2_DB, desc="tolerance", unit="level"):
    ds = datasets[db]
    for scen in SCENARIOS:
        for method in C_METHODS:
            arr = c_trajectories(method, ds, scen, db, N_C_PLOT)
            if arr is None:
                print(f"  {method:10s} [{scen} | {db:+g} dB]: checkpoint missing -- skipped")
                continue
            c_curves[(method, scen, db)] = arr
            c_rows += [{"model": method, "scenario": scen, "inv_r2_db": db, "t": t + 1,
                        "c_mean": float(v)} for t, v in enumerate(arr.mean(axis=0))]
        if INCLUDE_REKF:
            c_rows += [{"model": "REKF", "scenario": scen, "inv_r2_db": db, "t": t + 1,
                        "c_mean": float(c_star[(scen, db)])} for t in range(T)]

c_df = pd.DataFrame(c_rows)
c_df.to_csv(os.path.join(CKPT_DIR, "tolerance_estimates.csv"), index=False)

if c_df.empty:
    print("no tolerance estimates to draw")
else:
    steps = np.arange(1, T + 1)
    fig = make_subplots(rows=len(SCENARIOS), cols=len(INV_R2_DB),
                        shared_xaxes=True, shared_yaxes=True,
                        horizontal_spacing=0.04, vertical_spacing=0.10,
                        subplot_titles=[f"{scen} | 1/R² = {db:+g} dB"
                                        for scen in SCENARIOS for db in INV_R2_DB])
    for row, scen in enumerate(SCENARIOS, start=1):
        for col, db in enumerate(INV_R2_DB, start=1):
            first = (row == 1 and col == 1)
            if INCLUDE_REKF:
                fig.add_trace(go.Scatter(
                    x=steps, y=np.full(T, c_star[(scen, db)]), mode="lines", name="REKF (constant)",
                    legendgroup="REKF", showlegend=first,
                    line=dict(color=COLORS["REKF"], width=2.6)), row=row, col=col)
            for method in C_METHODS:
                arr = c_curves.get((method, scen, db))
                if arr is None:
                    continue
                fig.add_trace(go.Scatter(
                    x=steps, y=arr.mean(axis=0), mode="lines", name=f"{method} RT-KalmanNet",
                    legendgroup=method, showlegend=first,
                    line=dict(color=COLORS[method], width=1.8)), row=row, col=col)
            fig.update_xaxes(title_text="t" if row == len(SCENARIOS) else None, row=row, col=col)
        # c is a KL budget constrained to [0, 1]; the extra 0.05 of headroom keeps a curve or a
        # c* sitting at the upper bound from being drawn on the frame and lost.
        fig.update_yaxes(title_text="tolerance ĉ", range=[0, 1.05], dtick=0.2,
                         row=row, col=1)

    fig.update_layout(
        title=f"Estimated tolerance over time — mean of {N_C_PLOT} test trajectories "
              f"(thick grey = REKF constant c*)",
        height=340 * len(SCENARIOS), width=330 * len(INV_R2_DB) + 160,
        template="plotly_white", hovermode="x unified")
    fig.show()

    html = os.path.join(CKPT_DIR, "tolerance_estimates.html")
    fig.write_html(html)
    saved = [html]
    try:                                   # PNG needs kaleido; skip silently if unavailable
        png = os.path.join(CKPT_DIR, "tolerance_estimates.png")
        fig.write_image(png, scale=2)
        saved.append(png)
    except Exception as exc:
        print(f"  (PNG export skipped: {type(exc).__name__})")
    print("saved:", ", ".join(saved), "and", os.path.join(CKPT_DIR, "tolerance_estimates.csv"))

tolerance:   0%|          | 0/4 [00:00<?, ?level/s]

  (PNG export skipped: ValueError)
saved: Results_task4\tolerance_estimates.html and Results_task4\tolerance_estimates.csv


## 13. Final summary table

One compact table over both scenarios, all noise levels and all five estimators. Every number is taken from Sections 9 and 11 — the MSE columns from the sweep, the timing columns from the benchmark — so nothing is recomputed here. Both sections now cover the whole sweep, so the timing columns are pivoted per noise level exactly like the MSE ones.

In [14]:
# Compact overview. MSE columns come from Sec. 9 (results_df), timing columns from Sec. 11
# (timing_df). No metric is recomputed.
pivot = results_df.pivot_table(index=["scenario", "model"], columns="inv_r2_db",
                               values="mse_db", sort=False)
pivot.columns = [f"MSE_dB @ {c:+g}dB" for c in pivot.columns]

final_summary = pivot.reset_index()
if not timing_df.empty:
    # One column per (metric, noise level): timing is now measured across the whole sweep, so a
    # merge on (scenario, model) alone would duplicate every row once per level.
    t_pivot = timing_df.pivot_table(
        index=["scenario", "model"], columns="inv_r2_db",
        values=["time_per_traj_s", "time_per_step_ms", "train_time_s"], sort=False)
    t_pivot.columns = [f"{metric} @ {db:+g}dB" for metric, db in t_pivot.columns]
    final_summary = final_summary.merge(t_pivot.reset_index(), on=["scenario", "model"], how="left")

final_summary["model"] = pd.Categorical(final_summary["model"], METHODS, ordered=True)
final_summary["scenario"] = pd.Categorical(final_summary["scenario"], SCENARIOS, ordered=True)
final_summary = final_summary.sort_values(["scenario", "model"]).reset_index(drop=True)

out = os.path.join(CKPT_DIR, "summary_task4_final.csv")
final_summary.to_csv(out, index=False)
print(f"Task 4 — final summary  ({len(SCENARIOS)} scenarios x {len(METHODS)} estimators x "
      f"{len(INV_R2_DB)} noise levels)   ->  {out}")
print(f"timing columns measured over {N_TIME_SEQ} test sequences at every noise level\n")
print(final_summary.to_string(index=False))
final_summary

Task 4 — final summary  (2 scenarios x 5 estimators x 4 noise levels)   ->  Results_task4\summary_task4_final.csv
timing columns measured over 10 test sequences at every noise level

scenario     model  MSE_dB @ -12.04dB  MSE_dB @ -6.02dB  MSE_dB @ +0dB  MSE_dB @ +10dB  time_per_traj_s @ -12.04dB  time_per_traj_s @ -6.02dB  time_per_traj_s @ +0dB  time_per_traj_s @ +10dB  time_per_step_ms @ -12.04dB  time_per_step_ms @ -6.02dB  time_per_step_ms @ +0dB  time_per_step_ms @ +10dB  train_time_s @ -12.04dB  train_time_s @ -6.02dB  train_time_s @ +0dB  train_time_s @ +10dB
    full       EKF            -6.0654          -13.2680       -19.5517        -29.6273                      0.1436                     0.1482                  0.1487                   0.1499                        1.436                       1.482                    1.487                     1.499                      NaN                     NaN                  NaN                   NaN
    full      REKF            -6.05

,scenario,model,MSE_dB @ -12.04dB,MSE_dB @ -6.02dB,MSE_dB @ +0dB,MSE_dB @ +10dB,time_per_traj_s @ -12.04dB,time_per_traj_s @ -6.02dB,time_per_traj_s @ +0dB,time_per_traj_s @ +10dB,time_per_step_ms @ -12.04dB,time_per_step_ms @ -6.02dB,time_per_step_ms @ +0dB,time_per_step_ms @ +10dB,train_time_s @ -12.04dB,train_time_s @ -6.02dB,train_time_s @ +0dB,train_time_s @ +10dB
0,full,EKF,-6.0654,-13.2680,-19.5517,-29.6273,0.1436,0.1482,0.1487,0.1499,1.436,1.482,1.487,1.499,NaN,NaN,NaN,NaN
1,full,REKF,-6.0517,-13.2558,-19.5441,-29.6213,0.5447,0.5214,0.5103,0.5065,5.447,5.214,5.103,5.065,NaN,NaN,NaN,NaN
2,full,KalmanNet,-6.3369,-13.2998,-19.5213,-29.5994,0.1301,0.1311,0.1321,0.1357,1.301,1.311,1.321,1.357,9.4,9.2,9.3,9.5
3,full,Original,-5.6449,-13.1347,-19.4030,-29.4719,0.6710,0.6672,0.6805,0.6931,6.710,6.672,6.805,6.931,1144.8,1084.4,1292.0,1136.4
4,full,Proposed,-6.0613,-13.2595,-19.5331,-29.5219,0.7200,0.7227,0.7228,0.7755,7.200,7.227,7.228,7.755,2859.3,2035.7,2280.0,2696.0
5,partial,EKF,-2.1931,-5.7489,-8.2910,-8.6671,0.1285,0.1286,0.1297,0.1331,1.285,1.286,1.297,1.331,NaN,NaN,NaN,NaN
6,partial,REKF,-1.8521,-4.1333,-3.9049,-14.4795,0.4143,0.4914,0.5404,0.5936,4.143,4.914,5.404,5.936,NaN,NaN,NaN,NaN
7,partial,KalmanNet,-6.3194,-12.5460,-16.5667,-19.0377,0.1301,0.1312,0.1273,0.1293,1.301,1.312,1.273,1.293,9.1,9.2,9.2,9.0
8,partial,Original,2.4525,-0.1755,-2.7315,-13.6885,0.6547,0.6587,0.6430,0.6497,6.547,6.587,6.430,6.497,1129.9,1037.8,1251.6,1092.3
9,partial,Proposed,-1.5927,-4.7494,-5.6585,-14.4714,0.5731,0.6294,0.7185,0.7964,5.731,6.294,7.185,7.964,1994.9,1984.6,2634.3,2531.4


## 14. Files Produced and How to Re-run Cheaply

Everything this notebook writes lands in `CKPT_DIR`:

| file | content |
|---|---|
| `results_long.csv` | one row per (model, scenario, noise level) |
| `per_sequence_metrics.csv` | per-trajectory MSE, the basis of the error bars |
| `timing_focus.csv` | inference and training times, one row per (model, scenario, noise level) |
| `<method>_<scenario>_R<db>dB.pt` | best-validation checkpoints, one per combination |
| `datasets/data_R<db>dB.pt` | the generated datasets |
| `training_history.json` | training/validation loss curves + training times |
| `tolerance_estimates.csv` / `.html` | estimated tolerance per case (Sec. 12) |
| `summary_task4_final.csv` | the final summary table (Sec. 13) |

**Re-running without retraining.** Set `RETRAIN = False` and `REGEN_DATA = False` in Sec. 2: the
sweep then reuses the cached datasets and checkpoints, and every table and figure is rebuilt from
disk in seconds. This is the path to take when only the plots change. Sec. 8's loss curves survive
that path too: the history is written to `training_history.json` when training runs and is read
back when it is skipped.